In [2]:
# 1. INSTALACIÓN DE HERRAMIENTAS
# El signo '!' le dice al sistema que descargue e instale estos paquetes.
!pip install chembl_webresource_client
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 59.4 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, PandasTools
from rdkit import Chem
from rdkit.Chem import PandasTools
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from chembl_webresource_client.new_client import new_client
import pandas as pd
from tqdm.auto import tqdm

#  Conectamos con la API
molecule = new_client.molecule

#Regla de Lipinski
# Peso entre 300-500 y LogP <= 5
#we obtain a library all molecules already filter
compounds_provider = molecule.filter(
    molecule_type='Small molecule',
    molecule_properties__full_mwt__range=[300, 500],
    molecule_properties__alogp__lte=5
).only('molecule_chembl_id', 'molecule_structures')


print("Descargando compuestos...")
# se descarga las primeras 50k moleculas q cumplan esas condiciones
random_subset = compounds_provider[0:50000]

# Convertir a DataFrame
compounds_list = []
for comp in tqdm(random_subset):
    # Verificamos que tenga estructura
    if comp['molecule_structures'] and 'canonical_smiles' in comp['molecule_structures']:
        compounds_list.append({
            'chembl_id': comp['molecule_chembl_id'],
            'smiles': comp['molecule_structures']['canonical_smiles']
        })

df_random = pd.DataFrame(compounds_list)

print(f"Descargados {len(df_random)} compuestos.")
print(df_random.head())



#
filename = 'libreria_screening_masiva.csv'
df_random.to_csv(filename, index=False)


Descargando compuestos...


  0%|          | 0/50000 [00:00<?, ?it/s]

Descargados 50000 compuestos.
      chembl_id                                            smiles
0    CHEMBL6329      Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl
1    CHEMBL6328   Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1
2  CHEMBL265667  Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1
3    CHEMBL6362      Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1
4  CHEMBL267864    Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1
------------------------------------------------------------
✅ ¡TERMINADO! Se ha creado el archivo: libreria_screening_masiva.csv
📊 Total de moléculas listas para investigar: 50000
------------------------------------------------------------
Muestra de los datos:
      chembl_id                                            smiles
0    CHEMBL6329      Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl
1    CHEMBL6328   Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1
2  CHEMBL265667  Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1
3    CHEMBL6362      Cc1ccc(C(=O)c2ccc(-n3ncc(

In [5]:
df_screening = pd.read_csv("libreria_screening_masiva.csv")
df_screening.head()

,chembl_id,smiles
0,CHEMBL6329,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl
1,CHEMBL6328,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1
2,CHEMBL265667,Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1
3,CHEMBL6362,Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1
4,CHEMBL267864,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1


For filtering the library and make sure all the molecules satisfy the rules of Lipinsky I created this function based on the code of talktutorial notebook T002, 2019 edition. I edited to suit my aims
https://projects.volkamerlab.org/teachopencadd/talktorials.html

In [6]:
def calculate_ro5_properties(smiles):
    """
    Test if input molecule (SMILES) fulfills Lipinski's rule of five.

    Parameters
    ----------
    smiles : str
        SMILES for a molecule.

    Returns
    -------
    pandas.Series
        Molecular weight, number of hydrogen bond acceptors/donor and logP value
        and Lipinski's rule of five compliance for input molecule.
    """
    # RDKit molecule from SMILES
    molecule = Chem.MolFromSmiles(smiles)
    # Calculate Ro5-relevant chemical properties
    n_hba = Descriptors.NumHAcceptors(molecule)
    n_hbd = Descriptors.NumHDonors(molecule)
    # Check if Ro5 conditions fulfilled
    ro5_fulfilled = (n_hba <= 10) and (n_hbd <= 5)
    # Return True if no more than one out of four conditions is violated
    return pd.Series(
        [n_hba, n_hbd, ro5_fulfilled],
        index=[ "n_hba", "n_hbd", "ro5_fulfilled"],
    )

The for rules are satisfy. We are sure that our compounds are drug-like

In [7]:
ro5_properties = df_screening["smiles"].apply(calculate_ro5_properties)
ro5_properties.head()

,n_hba,n_hbd,ro5_fulfilled
0,5,1,True
1,6,1,True
2,5,2,True
3,5,1,True
4,5,1,True


In [8]:
ro5_properties['ro5_fulfilled'].value_counts()

,count
ro5_fulfilled,
True,49070
False,930


In [9]:
df_con_propiedades = pd.concat([df_random, ro5_properties], axis=1)

# 3. Filtramos: Nos quedamos solo con los que cumplen la regla
df_final = df_con_propiedades[df_con_propiedades['ro5_fulfilled'] == True]

# (Opcional) Si quieres limpiar, puedes borrar la columna 'ro5_fulfilled' ya que ahora todos son True
cols_del = ['ro5_fulfilled', "n_hba", "n_hbd"]
df_final = df_final.drop(columns=cols_del)

# Verificamos cuántos han sobrevivido
print(f"Compuestos iniciales: {len(df_random)}")
print(f"Compuestos finales tras filtrar: {len(df_final)}")
print(df_final.head())

Compuestos iniciales: 50000
Compuestos finales tras filtrar: 49070
      chembl_id                                            smiles
0    CHEMBL6329      Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl
1    CHEMBL6328   Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1
2  CHEMBL265667  Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1
3    CHEMBL6362      Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1
4  CHEMBL267864    Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1


# Next step removing pains

I bases these blocks of code again on the talktutorials T003


The goal of this step is to filter the PAIN which are molecules that bind nonspecifically to a targeted protein, but to a many of them. This posses a serious problem of toxicity, and they musy be eliminated

The catalogs of pains is the one that we can find in CHEMBL

In [10]:
# initialize filter
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
catalog = FilterCatalog(params)

In [11]:
# search for PAINS
matches = []
clean = []
for index, row in tqdm(df_final.iterrows(), total=df_final.shape[0]):
    molecule = Chem.MolFromSmiles(row.smiles)
    entry = catalog.GetFirstMatch(molecule)  # Get the first matching PAINS
    if entry is not None:
        # store PAINS information
        matches.append(
            {
                "chembl_id": row.chembl_id,
                "rdkit_molecule": molecule,
                "pains": entry.GetDescription().capitalize(),
            }
        )
    else:
        # collect indices of molecules without PAINS
        clean.append(index)

matches = pd.DataFrame(matches)
df_nopain = df_final.loc[clean]  # keep molecules without PAINS

  0%|          | 0/49070 [00:00<?, ?it/s]

In [12]:
# NBVAL_CHECK_OUTPUT
print(f"Number of compounds with PAINS: {len(matches)}")
print(f"Number of compounds without PAINS: {len(df_nopain)}")

Number of compounds with PAINS: 2717
Number of compounds without PAINS: 46353


# Save the result

In [16]:
import os


ruta_carpeta = '/content/drive/MyDrive/Colab_Notebooks/lead_discovery/final_assignment/'

# nombre del archivo
nombre_archivo = 'filtered_screening_library.csv'

#  Crear la ruta completa (Carpeta + Nombre del archivo)
ruta_completa = os.path.join(ruta_carpeta, nombre_archivo)


# index=False es importante para que no se guarde una columna extra con los números 0, 1, 2...
df_nopain.to_csv(ruta_completa, index=False)

print(f"¡Listo! Archivo guardado en: {ruta_completa}")

¡Listo! Archivo guardado en: /content/drive/MyDrive/Colab_Notebooks/lead_discovery/final_assignment/filtered_screening_library.csv
